In [2]:
import weaviate
from weaviate.classes.config  import Configure
import requests, json



***Create Collection***

In [3]:
client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)


print(client.is_ready())  
print(client.get_meta())

True
{'grpcMaxMessageSize': 104858000, 'hostname': 'http://[::]:8080', 'modules': {'generative-anthropic': {'documentationHref': 'https://docs.anthropic.com/en/api/getting-started', 'name': 'Generative Search - Anthropic'}, 'generative-anyscale': {'documentationHref': 'https://docs.anyscale.com/endpoints/overview', 'name': 'Generative Search - Anyscale'}, 'generative-aws': {'documentationHref': 'https://docs.aws.amazon.com/bedrock/latest/APIReference/welcome.html', 'name': 'Generative Search - AWS'}, 'generative-cohere': {'documentationHref': 'https://docs.cohere.com/reference/chat', 'name': 'Generative Search - Cohere'}, 'generative-databricks': {'documentationHref': 'https://docs.databricks.com/en/machine-learning/foundation-models/api-reference.html#completion-task', 'name': 'Generative Search - Databricks'}, 'generative-friendliai': {'documentationHref': 'https://docs.friendli.ai/openapi/create-chat-completions', 'name': 'Generative Search - FriendliAI'}, 'generative-google': {'doc

In [1]:
collections = client.collections.delete("Question")
print(collections)

NameError: name 'client' is not defined

In [11]:


questions = client.collections.create(
    name="Question",
    vector_config=Configure.Vectors.text2vec_ollama(  # Configure the Ollama embedding integration
        api_endpoint="http://172.17.0.6:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="nomic-embed-text:latest",  # The model to use
    ),
    generative_config=Configure.Generative.ollama(  # Configure the Ollama generative integration
        api_endpoint="http://172.17.0.6:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="llama3.2",  # The model to use
    ),
)

client.close()  # Free up resources

In [12]:
import weaviate
import requests, json


client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)
resp = requests.get(
    "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
)
data = json.loads(resp.text)

questions = client.collections.use("Question")

with questions.batch.fixed_size(batch_size=200) as batch:
    for d in data:
        batch.add_object(
            {
                "answer": d["Answer"],
                "question": d["Question"],
                "category": d["Category"],
            }
        )
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break

failed_objects = questions.batch.failed_objects
if failed_objects:
    print(f"Number of failed imports: {len(failed_objects)}")
    print(f"First failed object: {failed_objects[0]}")

client.close()  # Free up resources

In [13]:
import weaviate
import json

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)
questions = client.collections.use("Question")

response = questions.query.near_text(
    query="biology",
    limit=3
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=2))

client.close()  # Free up resources

{
  "question": "In 1953 Watson & Crick built a model of the molecular structure of this, the gene-carrying substance",
  "category": "SCIENCE",
  "answer": "DNA"
}
{
  "question": "This organ removes excess glucose from the blood & stores it as glycogen",
  "answer": "Liver",
  "category": "SCIENCE"
}
{
  "question": "It's the only living mammal in the order Proboseidea",
  "answer": "Elephant",
  "category": "ANIMALS"
}


In [14]:
import weaviate

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)

questions = client.collections.use("Question")

response = questions.generate.near_text(
    query="biology",
    limit=2,
    grouped_task="Write a tweet with emojis about these facts."
)

print(response.generative.text)  # Inspect the generated text

client.close()  # Free up resources

"Did you know? 🧬 DNA is the molecule that holds our genes, and was famously built by Watson & Crick in 1953! 💡 Meanwhile, your liver works hard to filter out excess sugar, storing it as glycogen for energy. Kudos to this tiny but mighty organ! 💪 #ScienceFacts #DNA #Liver"
